# 05 - Build the official submission (Option B, per-subject)

Produces the two files the challenge is scored on:
`sub-05_ranked_candidate_list.npy` and `sub-06_ranked_candidate_list.npy`.

**Method:** Option B (winner) - BERG `eeg-things_eeg_2-vit_b_32` in silico EEG
used as per-video features, mapped to each subject's real EEG with
StandardScaler -> PCA(0.95) -> StandardScaler -> RidgeCV. Trained **per subject**
on that subject's own training split.

**Format rules (from the challenge PDF):**
- Each file: integer array, shape `102 x 102`, values are video IDs `1001..1102`
  (not column indices), ordered most -> least likely.
- Row `i` must correspond to the `i`-th recording in `tst/sub-XX_tst.npy` in its
  **original, unsorted order**. We therefore do NOT sort rows by labels here
  (that label-sorting only exists in the validation notebooks 02/04).
- Exact filenames only. Different name/shape/content cannot be scored.

**Rule compliance:** sub-05/06 are ranked using only (a) their own training data
to train the encoder and (b) their own test recordings for unsupervised
correlation matching. No sub-01..04 test recordings are used as templates.

In [2]:
# Windows UTF-8 fix + clean berg import (see notebook 03/04)
import builtins, sys
if not getattr(builtins, '_utf8_open_patched', False):
    _orig_open = builtins.open
    def _utf8_open(*a, **kw):
        mode = kw.get('mode', a[1] if len(a) > 1 else 'r')
        if 'b' not in mode and kw.get('encoding') is None:
            kw['encoding'] = 'utf-8'
        return _orig_open(*a, **kw)
    builtins.open = _utf8_open
    builtins._utf8_open_patched = True
for _m in [m for m in list(sys.modules) if m == 'berg' or m.startswith('berg.')]:
    del sys.modules[_m]

import os, glob
import numpy as np
import cv2
from scipy.signal import decimate
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

device: cuda


In [4]:
# ---------------- Config ----------------
BERG_DIR   = './brain-encoding-response-generator'
MODEL_ID   = 'eeg-things_eeg_2-vit_b_32'

STIM_DIR        = './stimuli_train'
STIM_TEST_DIR   = './stimuli_test'
FRAME_DIR       = './frames'
FRAME_TEST_DIR  = './frames_test'
TRAIN_DIR       = './trn'
TEST_DIR        = './tst'
GT_DIR          = './tst_stim'
CACHE_DIR       = './features_berg'
os.makedirs(CACHE_DIR, exist_ok=True)

SFREQ, DEC, N_FRAMES, IMG_SIZE = 500, 5, 8, 224
tst_ids = list(range(1001, 1103))          # the 102 test videos (same for all subjects)
N_TEST  = len(tst_ids)

In [5]:
# ---------------- Shared helpers ----------------
def video_for(stim_id, vids):
    # training videos are 4-digit zero-padded (0012.mp4); test videos are 1001.mp4.
    # Accept both the plain and the zero-padded filename.
    targets = {f'{int(stim_id)}.mp4', f'{int(stim_id):04d}.mp4'}
    hits = [v for v in vids if os.path.basename(v) in targets]
    assert hits, f'no video for stimulus {stim_id} (tried {targets})'
    return hits[0]

def dump_frames(stim_ids, stim_dir, out_dir, n_frames=N_FRAMES):
    vids = sorted(glob.glob(os.path.join(stim_dir, '*.mp4')))
    os.makedirs(out_dir, exist_ok=True)
    for sid in stim_ids:
        if len(glob.glob(os.path.join(out_dir, f'{int(sid):05d}_f*.jpg'))) >= n_frames:
            continue
        cap = cv2.VideoCapture(video_for(sid, vids))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        picks = np.linspace(0, max(total - 1, 0), n_frames).astype(int)
        for k, fno in enumerate(picks):
            path = os.path.join(out_dir, f'{int(sid):05d}_f{k:02d}.jpg')
            if os.path.exists(path):
                continue
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(fno))
            ok, frame = cap.read()
            if ok:
                cv2.imwrite(path, frame)
        cap.release()

def frames_for(sid, out_dir):
    return sorted(glob.glob(os.path.join(out_dir, f'{int(sid):05d}_f*.jpg')))

def load_rgb(path):
    return cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)

def zrows(A):
    return (A - A.mean(1, keepdims=True)) / (A.std(1, keepdims=True) + 1e-8)

In [6]:
# ---------------- Load BERG + feature extractor ----------------
from berg import BERG
berg = BERG(berg_dir=BERG_DIR)
# subject index for the BERG model is fixed; we always use its subject-1 weights
# as the stimulus feature extractor (it is a stimulus descriptor, not our EEG).
eeg_model = berg.get_encoding_model(MODEL_ID, subject=1, device=DEVICE)

def to_berg_stimulus(rgb_images):
    out = []
    for im in rgb_images:
        sq = cv2.resize(im, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        out.append(np.transpose(sq, (2, 0, 1)))
    return np.stack(out).astype('uint8')

def berg_response(rgb_images):
    out = np.asarray(berg.encode(eeg_model, to_berg_stimulus(rgb_images)))
    if out.ndim == 4:
        out = out.mean(axis=1)          # average 4 repetitions -> (n, 63, 140)
    return out

def build_features(ids, frame_dir, cache_name):
    cache_path = os.path.join(CACHE_DIR, cache_name)
    if os.path.exists(cache_path):
        d = np.load(cache_path)
        if np.array_equal(d['ids'], np.asarray(ids)):
            return d['X']
    rows = []
    for sid in ids:
        paths = frames_for(sid, frame_dir)
        assert paths, f'no frames for stimulus {sid} in {frame_dir}'
        resp = berg_response([load_rgb(p) for p in paths])
        rows.append(resp.reshape(resp.shape[0], -1).mean(axis=0))
    X = np.stack(rows)
    np.savez_compressed(cache_path, X=X, ids=np.asarray(ids))
    return X

c:\Users\Tharun's Alien\Desktop\FruitBasket-SchubKI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1017 21:15:32.321000 27492 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Model loaded on cuda for subject 1


In [7]:
# ---------------- Test features: identical 102 videos for every subject ----------------
dump_frames(tst_ids, STIM_TEST_DIR, FRAME_TEST_DIR)
X_tst = build_features(tst_ids, FRAME_TEST_DIR, 'berg_tst_shared.npz')   # (102, 8820)
print('shared test features:', X_tst.shape)

shared test features: (102, 8820)


## Core routine: rank the 102 test videos for one subject (ORIGINAL order, no label sorting)

In [9]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import RidgeCV

def rank_for_subject(sub):
    """Train Option B on this subject's training split and rank the 102 test
    videos for each of its test recordings, in the ORIGINAL unsorted tst order.
    Returns ranked (102, 102) of video IDs, best-first."""
    # --- real training EEG for this subject ---
    trn = np.load(f'{TRAIN_DIR}/sub-{sub:02d}_trn.npz')
    trn_ids = np.array(sorted(trn.files, key=int), dtype=int)
    trn_eeg = np.stack([trn[str(i)] for i in trn_ids])
    trn_eeg = decimate(trn_eeg, DEC, axis=-1, ftype='fir', zero_phase=True)
    n_chan, n_time = trn_eeg.shape[1], trn_eeg.shape[2]

    # --- real test EEG (original, unsorted order) ---
    tst_eeg = np.load(f'{TEST_DIR}/sub-{sub:02d}_tst.npy')
    tst_eeg = decimate(tst_eeg, DEC, axis=-1, ftype='fir', zero_phase=True)

    # --- BERG features: train (this subject's videos) + shared test ---
    dump_frames(trn_ids, STIM_DIR, FRAME_DIR)
    X_trn = build_features(trn_ids, FRAME_DIR, f'berg_trn_sub-{sub:02d}.npz')

    # --- train encoder: BERG features -> real EEG ---
    sc  = StandardScaler().fit(X_trn)
    pca = PCA(n_components=0.95, svd_solver='full').fit(sc.transform(X_trn))
    X_trn_p = pca.transform(sc.transform(X_trn))
    sc2 = StandardScaler().fit(X_trn_p)
    model = RidgeCV(alphas=np.logspace(-3, 6, 20))
    model.fit(sc2.transform(X_trn_p), trn_eeg.reshape(len(trn_ids), -1))

    # --- predict EEG for the 102 test videos ---
    X_tst_s = sc2.transform(pca.transform(sc.transform(X_tst)))
    pred = model.predict(X_tst_s).reshape(N_TEST, n_chan, n_time)

    # --- match: measured recording i vs predicted video j (NO row sorting) ---
    M = zrows(tst_eeg[:, :, 5:60].reshape(N_TEST, -1))     # measured, original order
    P = zrows(pred[:, :, 5:60].reshape(N_TEST, -1))        # predicted, video order
    S = M @ P.T / M.shape[1]                                # (102 recordings, 102 videos)

    video_ids = np.array(tst_ids)
    ranked = video_ids[np.argsort(-S, axis=1)]             # (102, 102) ids best-first
    return ranked.astype(np.int64), model, (sc, pca, sc2)

## Format + score self-check on sub-01 (PDF-recommended)
Run the exact same submission routine on sub-01 and score against its ground
truth. This confirms the shape/content/order are correct before we trust 05/06.

In [10]:
ranked01, _, _ = rank_for_subject(1)
assert ranked01.shape == (102, 102), ranked01.shape
assert ranked01.min() >= 1001 and ranked01.max() <= 1102
assert all(len(set(row)) == 102 for row in ranked01), 'each row must list all 102 ids once'

gt01 = np.load(f'{GT_DIR}/sub-01_tst_stim.npy')            # true id per recording (orig order)
top1 = (ranked01[:, 0] == gt01).mean()
top5 = (ranked01[:, :5] == gt01[:, None]).any(axis=1).mean()
print('sub-01 self-check (ORIGINAL order):')
print(f'  shape {ranked01.shape}, ids in [{ranked01.min()},{ranked01.max()}]')
print(f'  top-1 {top1:.4f}   top-5 {top5:.4f}   (chance {1/102:.4f} / {5/102:.4f})')

sub-01 self-check (ORIGINAL order):
  shape (102, 102), ids in [1001,1102]
  top-1 0.0490   top-5 0.2157   (chance 0.0098 / 0.0490)


## Generate and save the official held-out submissions (sub-05, sub-06)

In [11]:
for sub in (5, 6):
    ranked, _, _ = rank_for_subject(sub)
    assert ranked.shape == (102, 102)
    assert ranked.min() >= 1001 and ranked.max() <= 1102
    assert all(len(set(row)) == 102 for row in ranked)
    fname = f'sub-{sub:02d}_ranked_candidate_list.npy'
    np.save(fname, ranked)
    print(f'saved {fname}  shape {ranked.shape}  dtype {ranked.dtype}')

Encoding EEG responses: 100%|██████████| 1/1 [00:00<00:00,  1.79it/s, Encoded images=8, Total images=8]


saved sub-05_ranked_candidate_list.npy  shape (102, 102)  dtype int64


Encoding EEG responses: 100%|██████████| 1/1 [00:00<00:00,  2.46it/s, Encoded images=8, Total images=8]


saved sub-06_ranked_candidate_list.npy  shape (102, 102)  dtype int64


In [12]:
# Final sanity read-back of the two submission files
for sub in (5, 6):
    a = np.load(f'sub-{sub:02d}_ranked_candidate_list.npy')
    print(f'sub-{sub:02d}: shape {a.shape}, dtype {a.dtype}, '
          f'min {a.min()}, max {a.max()}, row0[:5] {a[0,:5]}')

sub-05: shape (102, 102), dtype int64, min 1001, max 1102, row0[:5] [1075 1049 1012 1077 1066]
sub-06: shape (102, 102), dtype int64, min 1001, max 1102, row0[:5] [1052 1089 1085 1019 1051]


## Notes
- The two saved files are the official deliverables. Commit them to the repo.
- Everything is per-subject: encoder trained on sub-XX training EEG; ranking
  uses sub-XX's own test recordings for unsupervised matching. No cross-subject
  test templates are used.
- BERG model used as feature extractor: `eeg-things_eeg_2-vit_b_32` (record this
  exact name + version in the report; public weights need not be uploaded).